Import Datasets for BDC from WRDS


This file is used to extract data from WRDS
1. daily  trade data = crsp
2. quarterly nav = compustat
3. interest rates and credit spread = frb (federal reserve board on wrds)

Using the files above, create df_event_study

df_event_study = create dataset for each nav anouncement date 4 event records.
-1 = the baseline trade record (immediately before nav announcement)
0 = day of announcement
1 and 2 are the trade dates the day after subsequently.

In [12]:
import wrds
import pandas as pd
import numpy as np
import duckdb as dd
import statsmodels.formula.api as smf
db = wrds.Connection()

Loading library list...
Done


In [13]:
# db.list_libraries()
# # db.list_tables('crsp_q_mutualfunds')
# # db.describe_table('crsp_q_mutualfunds','monthly_returns')
# db.list_tables('crsp_a_stock')
# db.describe_table('crsp_a_stock','msf')

# db.list_libraries()

In [ ]:
# # 1. df_trade
# query = """
# SELECT 
#     d.date,
#     d.permno,
#     n.ticker,
#     d.prc AS price,
#     d.ret,
#     d.vol,
#     d.shrout
# FROM crsp.dsf d
# JOIN crsp.dsenames n
#     ON d.permno = n.permno
# WHERE n.ticker IN ('ARCC','MAIN')
#   AND d.date BETWEEN n.namedt AND n.nameendt
#   AND d.date >= '2015-12-01'
# ORDER BY d.permno, d.date"""

# df_trade = db.raw_sql(query)

# df_trade.loc[:, 'trade_date'] = pd.to_datetime(df_trade['date'])
# df_trade.head()

# df_trade.to_csv("df_trade.csv", index=False)


In [15]:
# 1. df_trade with excess return (using df_trade above - join to crsp.dsi*)

query = """
SELECT 
    d.date,
    d.permno,
    n.ticker,
    d.prc AS price,
    d.ret,
    d.vol,
    d.shrout,
    m.vwretd AS mkt_ret,
    d.ret - m.vwretd AS excess_ret
FROM crsp.dsf d
JOIN crsp.dsenames n
    ON d.permno = n.permno
JOIN crsp.dsi m
    ON d.date = m.date
WHERE n.ticker IN ('ARCC','MAIN')
  AND d.date BETWEEN n.namedt AND n.nameendt
  AND d.date >= '2015-12-01'
ORDER BY d.permno, d.date
"""

df_trade = db.raw_sql(query)

df_trade.loc[:, "trade_date"] = pd.to_datetime(df_trade["date"])
df_trade.head()


,date,permno,ticker,price,ret,vol,shrout,mkt_ret,excess_ret,trade_date
0,2015-12-01,90401,ARCC,15.87,0.003161,1425885.0,314469.0,0.009508,-0.006347,2015-12-01
1,2015-12-02,90401,ARCC,15.73,-0.008822,1233942.0,314469.0,-0.010722,0.0019,2015-12-02
2,2015-12-03,90401,ARCC,15.68,-0.003179,1548635.0,314469.0,-0.014317,0.011138,2015-12-03
3,2015-12-04,90401,ARCC,15.76,0.005102,1454772.0,314469.0,0.016454,-0.011352,2015-12-04
4,2015-12-07,90401,ARCC,15.36,-0.025381,1968566.0,314469.0,-0.009769,-0.015612,2015-12-07


In [16]:
# 2. df_rate

query = """
SELECT 
    date,
    dgs10,  
    dbaa, 
    daaa,
    (dbaa - dgs10) AS spread
FROM frb.rates_daily
WHERE cast(date as date) >= '2015-12-01'
ORDER BY date
"""

df_rate = db.raw_sql(query)

df_rate.loc[:, 'nav_date'] = pd.to_datetime(df_rate['date'])
df_rate = df_rate.drop(columns=['date'])

df_rate.head()

,dgs10,dbaa,daaa,spread,nav_date
0,2.15,5.36,3.92,3.21,2015-12-01
1,2.18,5.35,3.89,3.17,2015-12-02
2,2.33,5.51,4.06,3.18,2015-12-03
3,2.28,5.43,3.99,3.15,2015-12-04
4,<NA>,<NA>,<NA>,<NA>,2015-12-05


In [17]:
# 3. df nav |  calculate nav_ret

query = """SELECT
tic as ticker,
    gvkey,
    datadate,
    rdq,
    seqq,
    cshoq,
    (seqq / cshoq) AS nav
FROM comp.fundq
WHERE tic in  ('MAIN', 'ARCC')
AND datadate >= '2015-06-01'
ORDER BY datadate;"""

df_nav = db.raw_sql(query)

df_nav.loc[:, 'nav_date'] = pd.to_datetime(df_nav['datadate'])
df_nav.loc[:, 'announcement_date'] = pd.to_datetime(df_nav['rdq'])
df_nav = df_nav.drop(columns=['rdq', 'datadate'])

# sort by dates prior to calculating nav returns
df_nav = df_nav.sort_values(['ticker', 'announcement_date'])

df_nav.loc[:, 'nav_ret'] = df_nav.groupby('ticker')['nav'].pct_change()

df_nav.head()


,ticker,gvkey,seqq,cshoq,nav,nav_date,announcement_date,nav_ret
1,ARCC,161966,5282.441,314.469,16.797971,2015-06-30,2015-08-04,<NA>
3,ARCC,161966,5279.802,314.469,16.789579,2015-09-30,2015-11-04,-0.0005
4,ARCC,161966,5173.332,314.347,16.457393,2015-12-31,2016-02-24,-0.019785
6,ARCC,161966,5179.944,313.954,16.499054,2016-03-31,2016-05-04,0.002531
8,ARCC,161966,5218.041,313.954,16.6204,2016-06-30,2016-08-03,0.007355


In [18]:
# 4. df_trade daily + CRSP market return

query = """
SELECT 
    d.date as trade_date,
    d.permno,
    n.ticker,
    d.prc AS price,
    d.ret,
    d.vol,
    d.shrout,
    m.vwretd as mkt_ret,
    d.ret - m.vwretd as excess_ret
FROM crsp.dsf d
JOIN crsp.dsenames n
    ON d.permno = n.permno
JOIN crsp.dsi m
    ON d.date = m.date
WHERE n.ticker IN ('ARCC','MAIN')
  AND d.date BETWEEN n.namedt AND n.nameendt
  AND d.date >= '2015-12-01'
ORDER BY d.permno, d.date
"""

df_market_return = db.raw_sql(query)
df_market_return.head()


,trade_date,permno,ticker,price,ret,vol,shrout,mkt_ret,excess_ret
0,2015-12-01,90401,ARCC,15.87,0.003161,1425885.0,314469.0,0.009508,-0.006347
1,2015-12-02,90401,ARCC,15.73,-0.008822,1233942.0,314469.0,-0.010722,0.0019
2,2015-12-03,90401,ARCC,15.68,-0.003179,1548635.0,314469.0,-0.014317,0.011138
3,2015-12-04,90401,ARCC,15.76,0.005102,1454772.0,314469.0,0.016454,-0.011352
4,2015-12-07,90401,ARCC,15.36,-0.025381,1968566.0,314469.0,-0.009769,-0.015612


In [19]:
df_trade.info()
df_rate.info()
df_nav.info()   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4572 entries, 0 to 4571
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        4572 non-null   string        
 1   permno      4572 non-null   Int64         
 2   ticker      4572 non-null   string        
 3   price       4572 non-null   Float64       
 4   ret         4572 non-null   Float64       
 5   vol         4572 non-null   Float64       
 6   shrout      4572 non-null   Float64       
 7   mkt_ret     4572 non-null   Float64       
 8   excess_ret  4572 non-null   Float64       
 9   trade_date  4572 non-null   datetime64[ns]
dtypes: Float64(6), Int64(1), datetime64[ns](1), string(2)
memory usage: 388.6 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3363 entries, 0 to 3362
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   dgs10     2300 non-null   Float6

In [26]:
# trade_nav_df (trade + nav using ASOF)

query = """
SELECT 
    dt.ticker,
    dt.trade_date,
    dt.price,
    dt.ret,
    dn.nav ,
    dn.announcement_date,
    dn.nav_ret,
    dt.excess_ret
FROM df_trade dt
ASOF JOIN df_nav dn 
    ON UPPER(dt.ticker) = UPPER(dn.ticker) 
   AND dt.trade_date >= dn.announcement_date
ORDER BY dt.ticker, dt.trade_date
"""

trade_nav_df = dd.query(query).to_df()

cols_to_keep = [
    'ticker', 
    'announcement_date', 
    'trade_date', 
    'nav', 
    'nav_ret', 
    'ret', 
    'price',
    'excess_ret'
]

df_event_study_stg = trade_nav_df[cols_to_keep].copy()

df_event_study_stg.to_csv('df_event_study_stg.csv', index=False)


In [27]:
df_event_study_stg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4572 entries, 0 to 4571
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ticker             4572 non-null   object        
 1   announcement_date  4572 non-null   datetime64[ns]
 2   trade_date         4572 non-null   datetime64[ns]
 3   nav                4572 non-null   float64       
 4   nav_ret            4572 non-null   float64       
 5   ret                4572 non-null   float64       
 6   price              4572 non-null   float64       
 7   excess_ret         4572 non-null   float64       
dtypes: datetime64[ns](2), float64(5), object(1)
memory usage: 285.9+ KB


# the df_panel is a balanced panel event study datasets.
# we have 74 unique events + ticker. 
# each event has 4 rows.
# thats makes 74 * 4 = 296 rows for our panel study.

In [48]:
# df_panel balanced panel with day_from_event from -1 to +2

import duckdb as dd

df_panel = dd.sql("""
WITH numbered AS (
    SELECT
        ticker,
        announcement_date,
        trade_date,
        nav,
        nav_ret,
        ret,
        price,
        excess_ret,
        ROW_NUMBER() OVER (
            PARTITION BY ticker, announcement_date
            ORDER BY trade_date
        ) - 1 AS day_from_event
    FROM df_event_study_stg
),

post_event AS (
    SELECT
        ticker,
        announcement_date,
        trade_date,
        nav,
        nav_ret,
        ret,
        price,
        day_from_event
    FROM numbered
    WHERE day_from_event IN (0, 1, 2)
),

events AS (
    SELECT DISTINCT
        ticker,
        announcement_date
    FROM df_event_study_stg
),

pre_event AS (
    SELECT
        e.ticker,
        e.announcement_date,
        p.trade_date,
        p.nav,
        p.nav_ret,
        p.ret,
        p.price,
        -1 AS day_from_event
    FROM events e
    LEFT JOIN LATERAL (
        SELECT
            s.trade_date,
            s.nav,
            s.nav_ret,
            s.ret,
            s.price
        FROM df_event_study_stg s
        WHERE s.ticker = e.ticker
          AND s.trade_date < e.announcement_date
        ORDER BY s.trade_date DESC
        LIMIT 1
    ) p ON TRUE
),

event_panel_raw AS (
    SELECT * FROM pre_event
    UNION ALL
    SELECT * FROM post_event
),

valid_events AS (
    SELECT
        ticker,
        announcement_date
    FROM event_panel_raw
    GROUP BY ticker, announcement_date
    HAVING COUNT(DISTINCT day_from_event) = 4
)

SELECT
    r.ticker,
    r.announcement_date,
    r.trade_date,
    r.nav,
    r.nav_ret,
    r.ret,
    r.price,
    r.day_from_event
FROM event_panel_raw r
JOIN valid_events v
  ON r.ticker = v.ticker
 AND r.announcement_date = v.announcement_date
WHERE r.day_from_event IN (-1, 0, 1, 2)
ORDER BY r.ticker, r.announcement_date, r.day_from_event
""").df()

df_panel.to_csv('df_panel.csv', index=False)
df_panel.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296 entries, 0 to 295
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ticker             296 non-null    object        
 1   announcement_date  296 non-null    datetime64[ns]
 2   trade_date         294 non-null    datetime64[ns]
 3   nav                294 non-null    float64       
 4   nav_ret            294 non-null    float64       
 5   ret                294 non-null    float64       
 6   price              294 non-null    float64       
 7   day_from_event     296 non-null    int64         
dtypes: datetime64[ns](2), float64(4), int64(1), object(1)
memory usage: 18.6+ KB


In [47]:
# df_panel with excess return

# df_panel balanced panel with day_from_event from -1 to +2
import duckdb as dd

df_panel_car = dd.sql("""
WITH numbered AS (
    SELECT
        ticker,
        announcement_date,
        trade_date,
        nav,
        nav_ret,
        ret,
        price,
        excess_ret,
        ROW_NUMBER() OVER (
            PARTITION BY ticker, announcement_date
            ORDER BY trade_date
        ) - 1 AS day_from_event
    FROM df_event_study_stg
),

post_event AS (
    SELECT
        ticker,
        announcement_date,
        trade_date,
        nav,
        nav_ret,
        ret,
        price,
        excess_ret,
        day_from_event
    FROM numbered
    WHERE day_from_event IN (0, 1, 2)
),

events AS (
    SELECT DISTINCT
        ticker,
        announcement_date
    FROM df_event_study_stg
),

pre_event AS (
    SELECT
        e.ticker,
        e.announcement_date,
        p.trade_date,
        p.nav,
        p.nav_ret,
        p.ret,
        p.price,
        p.excess_ret,
        -1 AS day_from_event
    FROM events e
    LEFT JOIN LATERAL (
        SELECT
            s.trade_date,
            s.nav,
            s.nav_ret,
            s.ret,
            s.price,
            s.excess_ret
        FROM df_event_study_stg s
        WHERE s.ticker = e.ticker
          AND s.trade_date < e.announcement_date
        ORDER BY s.trade_date DESC
        LIMIT 1
    ) p ON TRUE
),

event_panel_raw AS (
    SELECT * FROM pre_event
    UNION ALL
    SELECT * FROM post_event
),

valid_events AS (
    SELECT
        ticker,
        announcement_date
    FROM event_panel_raw
    GROUP BY ticker, announcement_date
    HAVING COUNT(DISTINCT day_from_event) = 4
)

SELECT
    r.ticker,
    r.announcement_date,
    r.trade_date,
    r.nav,
    r.nav_ret,
    r.ret,
    r.price,
    r.excess_ret,
    r.day_from_event
FROM event_panel_raw r
JOIN valid_events v
  ON r.ticker = v.ticker
 AND r.announcement_date = v.announcement_date
WHERE r.day_from_event IN (-1, 0, 1, 2)
ORDER BY r.ticker, r.announcement_date, r.day_from_event
""").df()

df_panel.to_csv("df_panel_with_excess_returns.csv", index=False)
df_panel.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296 entries, 0 to 295
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ticker             296 non-null    object        
 1   announcement_date  296 non-null    datetime64[ns]
 2   trade_date         294 non-null    datetime64[ns]
 3   nav                294 non-null    float64       
 4   nav_ret            294 non-null    float64       
 5   ret                294 non-null    float64       
 6   price              294 non-null    float64       
 7   excess_ret         294 non-null    float64       
 8   day_from_event     296 non-null    int64         
dtypes: datetime64[ns](2), float64(5), int64(1), object(1)
memory usage: 20.9+ KB


*********   Regression Event Study  *******

In [49]:
import pandas as pd
import statsmodels.api as sm

df_reg = df_panel.copy()
df_reg = df_reg[df_reg["day_from_event"].isin([-1, 0, 1, 2])].copy()
df_reg = df_reg.dropna(subset=["ret"])

df_reg["event_id"] = (
    df_reg["ticker"].astype(str) + "_" +
    pd.to_datetime(df_reg["announcement_date"]).dt.strftime("%Y-%m-%d")
)

df_reg = pd.get_dummies(df_reg, columns=["day_from_event"], prefix="day")

for col in ["day_0", "day_1", "day_2"]:
    if col not in df_reg.columns:
        df_reg[col] = 0

X = sm.add_constant(df_reg[["day_0", "day_1", "day_2"]].astype(float))
y = pd.to_numeric(df_reg["ret"], errors="coerce").astype(float)

valid = y.notna()
X = X.loc[valid]
y = y.loc[valid]
groups = df_reg.loc[valid, "event_id"]

model = sm.OLS(y, X).fit(
    cov_type="cluster",
    cov_kwds={"groups": groups}
)

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                 -0.008
Method:                 Least Squares   F-statistic:                    0.1616
Date:                Wed, 15 Apr 2026   Prob (F-statistic):              0.922
Time:                        16:01:31   Log-Likelihood:                 788.55
No. Observations:                 294   AIC:                            -1569.
Df Residuals:                     290   BIC:                            -1554.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0032      0.001      2.623      0.0

/var/folders/kz/730dlv6x48x44yckc_2pv1n80000gn/T/ipykernel_78994/507887183.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_reg["event_id"] = (


In [62]:
# excess return event study regression

import pandas as pd
import statsmodels.api as sm

# Keep the event window only
df_car = df_panel_car.copy()
df_car = df_car[df_car["day_from_event"].isin([-1, 0, 1, 2])].copy()

# Drop missing abnormal returns
df_car = df_car.dropna(subset=["excess_ret"])

# Event-day dummies with day -1 as the omitted baseline
df_car = pd.get_dummies(df_car, columns=["day_from_event"], prefix="day")

for col in ["day_0", "day_1", "day_2"]:
    if col not in df_car.columns:
        df_car[col] = 0

X = sm.add_constant(df_car[["day_0", "day_1", "day_2"]].astype(float))
y = pd.to_numeric(df_car["excess_ret"], errors="coerce").astype(float)

valid = y.notna()
X = X.loc[valid]
y = y.loc[valid]

model_car = sm.OLS(y, X).fit()
print(model_car.summary())


                            OLS Regression Results                            
Dep. Variable:             excess_ret   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.007
Method:                 Least Squares   F-statistic:                    0.3230
Date:                Wed, 15 Apr 2026   Prob (F-statistic):              0.809
Time:                        16:22:18   Log-Likelihood:                 826.85
No. Observations:                 294   AIC:                            -1646.
Df Residuals:                     290   BIC:                            -1631.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0025      0.002      1.460      0.1

In [61]:
# CAR event study using -1 to 2

# CAR event study

import pandas as pd
import statsmodels.api as sm

# Keep only complete event-window rows with usable abnormal returns
df_car_reg = (
    df_panel_car[df_panel_car["day_from_event"].isin([-1, 0, 1, 2])]
    .dropna(subset=["excess_ret"])
    .copy()
)

# Construct event-level CAR
car_event = (
    df_car_reg
    .groupby(["ticker", "announcement_date"], as_index=False)
    .agg(
        CAR=("excess_ret", "sum"),
        nav_ret=("nav_ret", "first")
    )
)

# Regression: CAR on NAV return
X = sm.add_constant(car_event["nav_ret"].astype(float))
y = car_event["CAR"].astype(float)

model_car = sm.OLS(y, X).fit()
print(model_car.summary())



                            OLS Regression Results                            
Dep. Variable:                    CAR   R-squared:                       0.088
Model:                            OLS   Adj. R-squared:                  0.075
Method:                 Least Squares   F-statistic:                     6.956
Date:                Wed, 15 Apr 2026   Prob (F-statistic):             0.0102
Time:                        16:18:06   Log-Likelihood:                 169.10
No. Observations:                  74   AIC:                            -334.2
Df Residuals:                      72   BIC:                            -329.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0117      0.003      3.914      0.0

In [60]:
# CAR event study

import pandas as pd
import statsmodels.api as sm

# Keep only complete event-window rows with usable abnormal returns
df_car_reg = (
    df_panel_car[df_panel_car["day_from_event"].isin([0, 1, 2])]
    .dropna(subset=["excess_ret"])
    .copy()
)

# Construct event-level CAR
car_event = (
    df_car_reg
    .groupby(["ticker", "announcement_date"], as_index=False)
    .agg(
        CAR=("excess_ret", "sum"),
        nav_ret=("nav_ret", "first")
    )
)

# Regression: CAR on NAV return
X = sm.add_constant(car_event["nav_ret"].astype(float))
y = car_event["CAR"].astype(float)

model_car = sm.OLS(y, X).fit()
print(model_car.summary())



                            OLS Regression Results                            
Dep. Variable:                    CAR   R-squared:                       0.013
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.9375
Date:                Wed, 15 Apr 2026   Prob (F-statistic):              0.336
Time:                        16:16:57   Log-Likelihood:                 175.45
No. Observations:                  74   AIC:                            -346.9
Df Residuals:                      72   BIC:                            -342.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0080      0.003      2.907      0.0

Interpretations:

There is weak evidence of abnormal return around the broader window only when day -1 is included.
There is little evidence that the announcement itself generates a strong abnormal return over days 0 to 2